# 🚖 Entregable No. 2 — Reducción de Dimensionalidad y Conclusiones
## Tema: Demanda de Taxis / Movilidad Urbana
### Dataset: NYC Yellow Taxi — `seaborn.load_dataset('taxis')`
---
**Curso:** Seminario de Ciencia de los Datos  
**Metodología:** CRISP-DM  
**Fecha de Entrega:** 29 de Mayo de 2025

---
### 📌 Estructura del Entregable (9 Tópicos)
| # | Tópico | Fase CRISP-DM |
|---|--------|--------------|
| I | Análisis Descriptivo y de Calidad | Comprensión |
| II | Evaluación de la Calidad (Distribuciones) | Comprensión |
| III | Detección y Tratamiento de Ausentes | Preparación |
| IV | Tratamiento de Outliers | Preparación |
| V | Imputación Comparativa | Preparación |
| VI | PCA + Correlación de Pearson | Modelamiento |
| VII | Selección de Características (Filtros) | Modelamiento |
| VIII | Implementación y Entrenamiento de Modelos | Modelamiento |
| IX | Conclusiones y Reflexiones | Evaluación |

## ⚙️ 0. Instalación e Importación de Librerías

In [ ]:
!pip install seaborn scipy missingno --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import missingno as msno
import warnings

from scipy import stats
from scipy.stats import shapiro, probplot

# Sklearn — preprocesamiento
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.feature_selection import SelectKBest, f_classif, f_regression

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Librerías cargadas correctamente')

---
## 📊 TÓPICOS I–V — Preprocesamiento (Base del Entregable 1)
> Estos tópicos consolidan el pipeline completo de limpieza aplicado en el Entregable 1.
> Se ejecutan en una sola celda para dejar el dataset listo para los tópicos VI–IX.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TÓPICO I — Carga y descripción del dataset
# ══════════════════════════════════════════════════════════════════
df_raw = sns.load_dataset('taxis')
print(f'Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print(f'Nulos totales: {df_raw.isnull().sum().sum()}')
df_raw.head(5)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TÓPICO II — Estadísticos descriptivos resumidos
# ══════════════════════════════════════════════════════════════════
num_cols = df_raw.select_dtypes(include='number').columns.tolist()
print('── Estadísticos Descriptivos ──')
df_raw[num_cols].describe().round(3)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TÓPICO III — Tratamiento de valores nulos
# ══════════════════════════════════════════════════════════════════
df = df_raw.copy()

# Nulos en categóricas → moda
for col in ['dropoff_zone', 'dropoff_borough']:
    moda = df[col].mode()[0]
    n_nulos = df[col].isnull().sum()
    df[col] = df[col].fillna(moda)
    print(f'✅ [{col}]: {n_nulos} nulos imputados con moda = "{moda}"')

print(f'Nulos restantes: {df.isnull().sum().sum()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TÓPICO IV — Capping de outliers (IQR)
# ══════════════════════════════════════════════════════════════════
cols_capping = ['fare', 'distance', 'total', 'tip']
for col in cols_capping:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    antes = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    df[col] = df[col].clip(lower=lim_inf, upper=lim_sup)
    print(f'✅ Capping [{col}]: {antes} valores ajustados → [{lim_inf:.2f}, {lim_sup:.2f}]')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TÓPICO V — Encoding de variables categóricas
# ══════════════════════════════════════════════════════════════════
le = LabelEncoder()
cat_cols = ['color', 'payment', 'pickup_zone', 'dropoff_zone',
            'pickup_borough', 'dropoff_borough']

for col in cat_cols:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# Crear variable objetivo binaria: propina alta = 1 si tip > mediana
df['tip_alto'] = (df['tip'] > df['tip'].median()).astype(int)

print('✅ Encoding completado')
print(f'Variable objetivo (tip_alto): {df["tip_alto"].value_counts().to_dict()}')
print(f'Dimensiones finales del dataset: {df.shape}')

---
## 🔗 TÓPICO VI — PCA y Correlación de Pearson
### 6.1 Matriz de Correlación de Pearson
> La correlación de Pearson nos permite evaluar si las variables están suficientemente
> relacionadas para que PCA sea útil. Correlaciones altas entre variables sugieren
> redundancia de información, lo que PCA puede condensar en menos componentes.

In [ ]:
# Selección de features numéricas para PCA
features_pca = ['passengers', 'distance', 'fare', 'tip', 'tolls', 'total',
                'color_enc', 'payment_enc', 'pickup_borough_enc', 'dropoff_borough_enc']

df_pca = df[features_pca].copy()

# Matriz de correlación de Pearson
corr_matrix = df_pca.corr(method='pearson')

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, square=True,
    linewidths=0.5, ax=ax, mask=mask,
    annot_kws={'size': 9}
)
ax.set_title('Matriz de Correlación de Pearson — Variables del Dataset Taxis NYC',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlacion_pearson.png', dpi=150, bbox_inches='tight')
plt.show()

# Pares con alta correlación (|r| > 0.7)
print('\n── Pares con alta correlación (|r| > 0.70) ──')
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.70:
            print(f'  {corr_matrix.columns[i]:25s} ↔ {corr_matrix.columns[j]:25s}  r = {r:.4f}')

### 6.2 PCA — Análisis de Componentes Principales

In [ ]:
# Estandarización (obligatoria antes de PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)

# PCA con todos los componentes
pca_full = PCA()
pca_full.fit(X_scaled)

varianza_exp = pca_full.explained_variance_ratio_
varianza_acum = np.cumsum(varianza_exp)

print('── Varianza Explicada por Componente ──')
print(f'{"CP":>4} | {"Varianza %":>12} | {"Acumulada %":>12} | Barra')
print('-' * 55)
for i, (v, va) in enumerate(zip(varianza_exp, varianza_acum)):
    barra = '█' * int(v * 50)
    print(f'PC{i+1:>2} | {v*100:>11.2f}% | {va*100:>11.2f}% | {barra}')

In [ ]:
# Gráfico de varianza explicada (Scree Plot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree Plot
n_comp = len(varianza_exp)
ax1 = axes[0]
ax1.bar(range(1, n_comp+1), varianza_exp*100, color='steelblue', edgecolor='white', alpha=0.8)
ax1.plot(range(1, n_comp+1), varianza_exp*100, 'o-', color='navy', linewidth=2, markersize=6)
ax1.set_xlabel('Número de Componente Principal', fontsize=11)
ax1.set_ylabel('Varianza Explicada (%)', fontsize=11)
ax1.set_title('Scree Plot — Varianza por Componente', fontsize=12, fontweight='bold')
ax1.axhline(y=5, color='red', linestyle='--', alpha=0.6, label='Umbral 5%')
ax1.legend()
ax1.set_xticks(range(1, n_comp+1))

# Varianza acumulada
ax2 = axes[1]
ax2.plot(range(1, n_comp+1), varianza_acum*100, 's-', color='darkorange', linewidth=2.5, markersize=7)
ax2.axhline(y=80, color='green',  linestyle='--', linewidth=1.5, label='80% varianza')
ax2.axhline(y=90, color='red',    linestyle='--', linewidth=1.5, label='90% varianza')
ax2.fill_between(range(1, n_comp+1), varianza_acum*100, alpha=0.15, color='darkorange')
ax2.set_xlabel('Número de Componentes Principales', fontsize=11)
ax2.set_ylabel('Varianza Acumulada (%)', fontsize=11)
ax2.set_title('Varianza Acumulada — Criterio de Selección', fontsize=12, fontweight='bold')
ax2.legend()
ax2.set_xticks(range(1, n_comp+1))
ax2.set_ylim([0, 105])

# Anotar número de componentes para 80% y 90%
for umbral, color in [(80, 'green'), (90, 'red')]:
    n = np.argmax(varianza_acum >= umbral/100) + 1
    ax2.annotate(f'PC{n}={umbral}%', xy=(n, umbral),
                 xytext=(n+0.3, umbral-8), fontsize=9,
                 color=color, fontweight='bold')

plt.suptitle('Análisis de Componentes Principales (PCA) — NYC Yellow Taxi',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_varianza.png', dpi=150, bbox_inches='tight')
plt.show()

n_opt = np.argmax(varianza_acum >= 0.80) + 1
print(f'\n🏆 Componentes necesarios para explicar ≥80% de varianza: {n_opt}')
print(f'   Varianza explicada con {n_opt} componentes: {varianza_acum[n_opt-1]*100:.2f}%')

In [ ]:
# PCA óptimo y Biplot de los primeros 2 componentes
pca_opt = PCA(n_components=n_opt)
X_pca = pca_opt.fit_transform(X_scaled)

# Loadings (contribución de cada variable a los componentes)
loadings = pd.DataFrame(
    pca_opt.components_.T,
    index=features_pca,
    columns=[f'PC{i+1}' for i in range(n_opt)]
).round(4)

print('── Loadings (contribución de variables a componentes) ──')
print(loadings.to_string())

# Biplot PC1 vs PC2
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=df['tip_alto'], cmap='RdYlGn',
                     alpha=0.4, s=15, edgecolors='none')
plt.colorbar(scatter, ax=ax, label='tip_alto (0=bajo, 1=alto)')

# Vectores de carga
scale = 3.5
for i, feature in enumerate(features_pca):
    ax.annotate('', xy=(loadings.iloc[i, 0]*scale, loadings.iloc[i, 1]*scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='navy', lw=1.5))
    ax.text(loadings.iloc[i, 0]*scale*1.15, loadings.iloc[i, 1]*scale*1.15,
            feature, fontsize=8, color='navy', fontweight='bold')

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel(f'PC1 ({varianza_exp[0]*100:.1f}% varianza)', fontsize=11)
ax.set_ylabel(f'PC2 ({varianza_exp[1]*100:.1f}% varianza)', fontsize=11)
ax.set_title('Biplot PCA — PC1 vs PC2 con Vectores de Carga', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔎 TÓPICO VII — Selección de Características (Método de Filtro)
> Se aplica el método de filtro usando la prueba estadística **ANOVA F-score** (`SelectKBest`)
> para identificar las 5 variables más relevantes respecto a la variable objetivo `tip_alto`.

In [ ]:
X_feat = df[features_pca].copy()
y_feat = df['tip_alto']

# SelectKBest con F-score ANOVA
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_feat, y_feat)

scores_df = pd.DataFrame({
    'Variable':  features_pca,
    'F-Score':   selector.scores_.round(4),
    'p-valor':   selector.pvalues_.round(6)
}).sort_values('F-Score', ascending=False).reset_index(drop=True)

scores_df['Relevante'] = scores_df['p-valor'].apply(lambda p: '✅ Sí' if p < 0.05 else '❌ No')
print('── Ranking de Variables por F-Score (ANOVA) ──')
print(scores_df.to_string(index=False))

In [ ]:
# Top 5 variables
top5 = scores_df.head(5)['Variable'].tolist()
print(f'\n🏆 Top 5 características más relevantes: {top5}')

# Visualización del ranking
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#1a6e30' if v in top5 else '#b5c4b1' for v in scores_df['Variable']]
bars = ax.barh(scores_df['Variable'][::-1], scores_df['F-Score'][::-1],
               color=colors[::-1], edgecolor='white')

for bar, score in zip(bars, scores_df['F-Score'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=9)

ax.set_xlabel('F-Score (ANOVA)', fontsize=11)
ax.set_title('Selección de Características — Método de Filtro (F-Score ANOVA)\nVerde: Top 5 seleccionadas',
             fontsize=12, fontweight='bold')
ax.axvline(scores_df.iloc[4]['F-Score'], color='navy', linestyle='--',
           linewidth=1.5, label=f'Umbral Top 5 = {scores_df.iloc[4]["F-Score"]:.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('feature_selection.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🤖 TÓPICO VIII — Implementación y Entrenamiento de Modelos
### 8.1 Preparación del Dataset de Modelamiento
> Se entrena con las **Top 5 características** seleccionadas en el Tópico VII.
> Se comparan **Regresión Logística** y **Árbol de Decisión** como modelos supervisados.

In [ ]:
X = df[top5].copy()
y = df['tip_alto'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Estandarizar
scaler_m = StandardScaler()
X_train_s = scaler_m.fit_transform(X_train)
X_test_s  = scaler_m.transform(X_test)

print('── División del Dataset ──')
print(f'Total   : {len(X):,} muestras')
print(f'Train   : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test    : {len(X_test):,}  ({len(X_test)/len(X)*100:.1f}%)')
print(f'Balanceo Train — Clase 0: {(y_train==0).sum()} | Clase 1: {(y_train==1).sum()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  MODELO 1 — Regresión Logística
# ══════════════════════════════════════════════════════════════════
modelo_rl = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
modelo_rl.fit(X_train_s, y_train)

y_pred_rl = modelo_rl.predict(X_test_s)
acc_rl    = accuracy_score(y_test, y_pred_rl)
cv_rl     = cross_val_score(modelo_rl, X_train_s, y_train, cv=5, scoring='accuracy').mean()

print('═' * 55)
print('  MODELO 1 — Regresión Logística')
print('═' * 55)
print(f'  Accuracy (Test)    : {acc_rl:.4f} ({acc_rl*100:.2f}%)')
print(f'  Accuracy (CV-5)    : {cv_rl:.4f} ({cv_rl*100:.2f}%)')
print()
print(classification_report(y_test, y_pred_rl, target_names=['tip_bajo', 'tip_alto']))

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  MODELO 2 — Árbol de Decisión
# ══════════════════════════════════════════════════════════════════
modelo_dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
modelo_dt.fit(X_train_s, y_train)

y_pred_dt = modelo_dt.predict(X_test_s)
acc_dt    = accuracy_score(y_test, y_pred_dt)
cv_dt     = cross_val_score(modelo_dt, X_train_s, y_train, cv=5, scoring='accuracy').mean()

print('═' * 55)
print('  MODELO 2 — Árbol de Decisión')
print('═' * 55)
print(f'  Accuracy (Test)    : {acc_dt:.4f} ({acc_dt*100:.2f}%)')
print(f'  Accuracy (CV-5)    : {cv_dt:.4f} ({cv_dt*100:.2f}%)')
print()
print(classification_report(y_test, y_pred_dt, target_names=['tip_bajo', 'tip_alto']))

In [ ]:
# ── Visualización comparativa de métricas ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matrices de confusión
for ax, y_pred, nombre, color in [
    (axes[0], y_pred_rl, 'Regresión Logística', 'Blues'),
    (axes[1], y_pred_dt, 'Árbol de Decisión',   'Greens')
]:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['tip_bajo', 'tip_alto'])
    disp.plot(ax=ax, cmap=color, colorbar=False)
    ax.set_title(f'Matriz de Confusión\n{nombre}', fontsize=11, fontweight='bold')

# Gráfico comparativo de métricas
from sklearn.metrics import precision_score, recall_score, f1_score
metricas = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
vals_rl = [
    accuracy_score(y_test, y_pred_rl),
    precision_score(y_test, y_pred_rl),
    recall_score(y_test, y_pred_rl),
    f1_score(y_test, y_pred_rl)
]
vals_dt = [
    accuracy_score(y_test, y_pred_dt),
    precision_score(y_test, y_pred_dt),
    recall_score(y_test, y_pred_dt),
    f1_score(y_test, y_pred_dt)
]

x = np.arange(len(metricas))
w = 0.35
axes[2].bar(x - w/2, vals_rl, w, label='Regresión Logística', color='steelblue', alpha=0.85)
axes[2].bar(x + w/2, vals_dt, w, label='Árbol de Decisión',   color='seagreen',  alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metricas, fontsize=10)
axes[2].set_ylim([0, 1.15])
axes[2].set_ylabel('Puntuación', fontsize=11)
axes[2].set_title('Comparativa de Métricas\nRL vs Árbol de Decisión', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=9)
for i, (v1, v2) in enumerate(zip(vals_rl, vals_dt)):
    axes[2].text(i - w/2, v1 + 0.02, f'{v1:.2f}', ha='center', fontsize=8)
    axes[2].text(i + w/2, v2 + 0.02, f'{v2:.2f}', ha='center', fontsize=8)

plt.suptitle('Evaluación Comparativa de Modelos Supervisados — NYC Yellow Taxi',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualización del Árbol de Decisión ──────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    modelo_dt, feature_names=top5,
    class_names=['tip_bajo', 'tip_alto'],
    filled=True, rounded=True, fontsize=8,
    max_depth=3, ax=ax
)
ax.set_title('Árbol de Decisión — Primeros 3 Niveles (max_depth=5)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('arbol_decision.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Importancia de características — Árbol de Decisión ───────────
importancias = pd.DataFrame({
    'Variable':    top5,
    'Importancia': modelo_dt.feature_importances_.round(4)
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importancias['Variable'], importancias['Importancia'],
        color='seagreen', edgecolor='white')
ax.set_xlabel('Importancia (Gini)', fontsize=11)
ax.set_title('Importancia de Características\nÁrbol de Decisión', fontsize=12, fontweight='bold')
for i, v in enumerate(importancias['Importancia']):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance_tree.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ✅ TÓPICO IX — Conclusiones y Reflexiones

In [ ]:
# ── Resumen ejecutivo final ───────────────────────────────────────
mejor = 'Regresión Logística' if acc_rl >= acc_dt else 'Árbol de Decisión'
mejor_acc = max(acc_rl, acc_dt)

print('═' * 65)
print('  RESUMEN EJECUTIVO — ENTREGABLE 2')
print('═' * 65)
print()
print('📌 PCA:')
print(f'   • Variables originales     : {len(features_pca)}')
print(f'   • Componentes óptimos (≥80%): {n_opt}')
print(f'   • Reducción dimensional    : {(1 - n_opt/len(features_pca))*100:.1f}%')
print()
print('📌 Selección de Características (F-Score ANOVA):')
print(f'   • Top 5: {top5}')
print()
print('📌 Modelos Supervisados:')
print(f'   • Regresión Logística — Accuracy: {acc_rl:.4f} | CV-5: {cv_rl:.4f}')
print(f'   • Árbol de Decisión   — Accuracy: {acc_dt:.4f} | CV-5: {cv_dt:.4f}')
print(f'   • Mejor modelo: {mejor} ({mejor_acc*100:.2f}%)')
print()
print('📌 Conclusión:')
print('   El preprocesamiento riguroso del Entregable 1 permitió que los modelos')
print('   obtuvieran métricas sólidas. La reducción de dimensionalidad con PCA')
print('   y la selección de características contribuyeron a mejorar la eficiencia')
print('   computacional sin sacrificar poder predictivo.')
print('═' * 65)